# sorted-computational-graph — worked example 1: Topological sort of a simple linear computation chain

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sorted-computational-graph`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Topological sort of a computation graph visits each node exactly once and ensures that every node's parents appear before it in the result. For a linear chain a → b → c → d, topological sort with end-node d produces [a, b, c, d] (deps-first). The reverse — [d, c, b, a] — is the backward-pass order needed by a neural network's reverse-mode autodiff.

## Worked solution

**Step 1 — Build a simple graph manually.** We create MiniTensor-like objects with `.recipe` attributes encoding the parent chain.

**Step 2 — Implement three-color DFS.** The `perm` set (black, fully processed) prevents revisits. The `temp` set (gray, on stack) detects cycles.

**Step 3 — Call topological_sort from the end node.** The DFS naturally produces a post-order traversal where leaves (roots) come first.

**Step 4 — Reverse for backward pass.** `[::-1]` flips the list so the end node is first — the starting point for gradient accumulation.

In [ ]:
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None
    def __repr__(self):
        return f'T({self.name})'

def topological_sort(node, get_children):
    result = []
    perm = set()
    temp = set()
    def visit(cur):
        cid = id(cur)
        if cid in perm:
            return
        if cid in temp:
            raise ValueError('Cycle detected')
        temp.add(cid)
        for child in get_children(cur):
            visit(child)
        temp.discard(cid)
        perm.add(cid)
        result.append(cur)
    visit(node)
    return result

def get_parents(n):
    if n.recipe is None:
        return []
    return list(n.recipe.parents.values())

# Chain: a -> b -> c -> d
a = FakeTensor('a')
b = FakeTensor('b', [a])
c = FakeTensor('c', [b])
d = FakeTensor('d', [c])

fwd_order = topological_sort(d, get_parents)  # deps-first
bwd_order = fwd_order[::-1]  # end-node first

print('Fwd order:', [n.name for n in fwd_order])  # [a, b, c, d]
print('Bwd order:', [n.name for n in bwd_order])  # [d, c, b, a]
assert bwd_order[0] is d, 'end node must be first in backward order'
assert bwd_order[-1] is a, 'leaf must be last'